# Week 2 — Analytical Thinking + Window Functions (Advanced Notebook)
## Overview
- This notebook demonstrates how analysts think using SQL
- We combine row-level + aggregated insights using window functions
- Each section includes: explanation → SQL → business insight → visualization

## Setup: Connect to DuckDB
- We use DuckDB as analytical engine
- JupySQL allows SQL inside notebook
- This creates a local analytical environment

In [ ]:
%load_ext sql
%sql duckdb:///week02_advanced.duckdb

## Load Data
- Load insurance dataset into DuckDB
- This becomes our analytical table
- We will query this table throughout

In [ ]:

%sql DROP TABLE IF EXISTS insurance;
%sql CREATE TABLE insurance AS 
SELECT * FROM read_csv_auto('/Users/max/mp/work_in_jupyter/insurance.csv');


## Preview Data
- Understand structure of dataset
- Identify key variables
- This is exploration, not analysis

In [ ]:

res = %sql SELECT * FROM insurance LIMIT 5
df = res.DataFrame()
df


In [ ]:

import matplotlib.pyplot as plt


## 1. Top-N per Region (Window Function)
- Business Question: Who are top customers per region?
- Use ROW_NUMBER with PARTITION
- Keeps row-level detail while ranking

In [ ]:

res = %sql
SELECT *
FROM (
    SELECT *, 
           ROW_NUMBER() OVER (PARTITION BY region ORDER BY charges DESC) AS rn
    FROM insurance
) t
WHERE rn <= 3

df = res.DataFrame()
df


### Business Insight
- Identifies VIP customers in each region
- Enables targeted marketing or pricing
- Helps allocate resources effectively

In [ ]:

df_group = df.groupby("region")["charges"].mean()

plt.figure()
df_group.plot(kind="bar")
plt.title("Top Customers Avg Charges by Region")
plt.xlabel("Region")
plt.ylabel("Charges")
plt.show()


## 2. Compare Customers to Regional Average
- Add context to each row
- Compare individual vs group
- This is key analytical thinking

In [ ]:

res = %sql
SELECT *, 
       AVG(charges) OVER (PARTITION BY region) AS regional_avg
FROM insurance

df = res.DataFrame()
df.head()


### Business Insight
- Shows whether customer is above or below norm
- Helps identify outliers
- Useful for pricing strategies

In [ ]:

diff = df.groupby("region")["regional_avg"].mean()

plt.figure()
diff.plot(kind="bar")
plt.title("Regional Average Charges")
plt.xlabel("Region")
plt.ylabel("Avg Charges")
plt.show()


## 3. Customers Above Regional Average
- Identify high-value customers
- Filter using window result
- Common real-world pattern

In [ ]:

res = %sql
SELECT *
FROM (
    SELECT *, AVG(charges) OVER (PARTITION BY region) AS regional_avg
    FROM insurance
) t
WHERE charges > regional_avg

df = res.DataFrame()
df.head()


### Business Insight
- Focus on customers generating more revenue
- Helps segmentation
- Supports retention strategies

In [ ]:

count = df.groupby("region").size()

plt.figure()
count.plot(kind="bar")
plt.title("Above Avg Customers by Region")
plt.xlabel("Region")
plt.ylabel("Count")
plt.show()


## 4. Contribution to Total Revenue
- Measure importance of each customer
- Use SUM window function
- Enables Pareto analysis

In [ ]:

res = %sql
SELECT *, charges / SUM(charges) OVER () AS pct_total
FROM insurance

df = res.DataFrame()
df.head()


### Business Insight
- Identify customers driving most revenue
- Supports prioritization
- Helps focus on high-impact users

In [ ]:

top = df.sort_values("pct_total", ascending=False).head(10)

plt.figure()
plt.bar(range(len(top)), top["pct_total"])
plt.title("Top Revenue Contributors")
plt.xlabel("Top Customers")
plt.ylabel("Contribution")
plt.show()


## 5. Running Total
- Shows accumulation of values
- Useful for trends
- Demonstrates progression

In [ ]:

res = %sql
SELECT age, charges,
       SUM(charges) OVER (ORDER BY age) AS running_total
FROM insurance

df = res.DataFrame()
df.head()


### Business Insight
- Helps understand cumulative trends
- Useful for forecasting
- Shows how values build up

In [ ]:

plt.figure()
plt.plot(df["age"], df["running_total"])
plt.title("Running Total by Age")
plt.xlabel("Age")
plt.ylabel("Running Total")
plt.show()


## 6. LAG / LEAD (Comparison)
- Compare neighboring rows
- Detect changes
- Used in time-series

In [ ]:

res = %sql
SELECT age, charges,
       LAG(charges) OVER (ORDER BY age) AS prev,
       LEAD(charges) OVER (ORDER BY age) AS next
FROM insurance

df = res.DataFrame()
df.head()


### Business Insight
- Detect changes between observations
- Useful for anomaly detection
- Supports trend analysis

In [ ]:

plt.figure()
plt.plot(df["age"], df["charges"])
plt.title("Charges by Age")
plt.xlabel("Age")
plt.ylabel("Charges")
plt.show()
